# 🍷 Wine Quality Prediction using K-Nearest Neighbors (KNN)
**Dataset:** Red Wine Quality (UCI Machine Learning Repository)  
**Algorithm:** K-Nearest Neighbors (KNN) Classifier  
**Objective:** Predict whether a wine is **Good** (quality ≥ 6) or **Bad** (quality < 6) based on physicochemical properties.

---
### 📋 Features:
| Feature | Description |
|---------|-------------|
| Fixed Acidity | Tartaric acid concentration (g/dm³) |
| Volatile Acidity | Acetic acid (vinegar taste indicator) |
| Citric Acid | Freshness and flavor enhancement |
| Residual Sugar | Sugar remaining after fermentation |
| Chlorides | Salt content |
| Free Sulfur Dioxide | SO₂ used as preservative |
| Total Sulfur Dioxide | Total SO₂ bound and free |
| Density | Density of the wine |
| pH | Acidity level (0–14) |
| Sulphates | Antimicrobial additive |
| Alcohol | Alcohol percentage (%) |

**Target:** quality_label → `Good` or `Bad`

## Step 1: Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

print('✅ All libraries imported successfully!')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## Step 2: Load and Explore the Dataset

In [ ]:
# Load dataset
df = pd.read_csv('winequality-red.csv')

print('📊 Dataset Shape:', df.shape)
print('\n📋 First 5 rows:')
df.head()

In [ ]:
# Basic statistics
print('📈 Statistical Summary:')
df.describe().round(3)

In [ ]:
# Check for missing values
print('🔍 Missing Values:')
print(df.isnull().sum())
print('\n✅ No missing values!' if df.isnull().sum().sum() == 0 else '⚠️ Missing values found!')

In [ ]:
# Quality score distribution
plt.figure(figsize=(8, 5))
quality_counts = df['quality'].value_counts().sort_index()
bars = plt.bar(quality_counts.index, quality_counts.values, 
               color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60','#1abc9c'],
               edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, quality_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(val), ha='center', fontsize=11, fontweight='bold')

plt.title('Wine Quality Score Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Quality Score (3–9)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.tight_layout()
plt.savefig('quality_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Most wines score 5 or 6 — moderate quality is most common.')

## Step 3: Data Visualization (EDA)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions
features = df.columns[:-1]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

colors = plt.cm.Set2(np.linspace(0, 1, len(features)))
for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].hist(df[feat], bins=30, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Value', fontsize=8)
    axes[i].set_ylabel('Frequency', fontsize=8)

# Hide last empty subplot
axes[-1].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Alcohol vs quality boxplot
plt.figure(figsize=(8, 5))
df.boxplot(column='alcohol', by='quality', grid=False)
plt.title('Alcohol Content by Wine Quality', fontsize=13, fontweight='bold')
plt.suptitle('')
plt.xlabel('Quality Score')
plt.ylabel('Alcohol (%)')
plt.tight_layout()
plt.show()
print('Higher quality wines tend to have higher alcohol content!')

## Step 4: Feature Engineering — Create Target Label

In [ ]:
# Create binary classification target
# Good: quality >= 6  |  Bad: quality < 6
df['quality_label'] = df['quality'].apply(lambda x: 'Good' if x >= 6 else 'Bad')

print('🏷️ Class Distribution:')
print(df['quality_label'].value_counts())
print(f'\nGood wines: {(df["quality_label"]=="Good").sum()} ({(df["quality_label"]=="Good").mean()*100:.1f}%)')
print(f'Bad wines:  {(df["quality_label"]=="Bad").sum()} ({(df["quality_label"]=="Bad").mean()*100:.1f}%)')

# Pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

counts = df['quality_label'].value_counts()
ax1.pie(counts, labels=counts.index, autopct='%1.1f%%',
        colors=['#2ecc71','#e74c3c'], startangle=90,
        explode=[0.05, 0.05], shadow=True)
ax1.set_title('Wine Quality: Good vs Bad', fontweight='bold')

ax2.bar(counts.index, counts.values, color=['#2ecc71','#e74c3c'],
        edgecolor='white', linewidth=1.5)
for i, (label, val) in enumerate(counts.items()):
    ax2.text(i, val + 10, str(val), ha='center', fontweight='bold')
ax2.set_title('Class Counts', fontweight='bold')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Prepare Features and Split Data

In [ ]:
# Separate features and target
X = df.drop(['quality', 'quality_label'], axis=1)
y = df['quality_label']

print('✅ Features shape:', X.shape)
print('✅ Target shape:', y.shape)
print('\nFeature columns:', list(X.columns))

In [ ]:
# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'🔀 Train size: {X_train.shape[0]} samples')
print(f'🔀 Test size:  {X_test.shape[0]} samples')
print(f'\nTrain class balance:\n{y_train.value_counts()}')
print(f'\nTest class balance:\n{y_test.value_counts()}')

## Step 6: Feature Scaling (StandardScaler)

In [ ]:
# KNN is distance-based → MUST scale features!
scaler = StandardScaler()

# Fit ONLY on train, transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Scaling complete!')
print(f'Mean before scaling: {X_train.mean().mean():.3f}')
print(f'Mean after  scaling: {X_train_scaled.mean():.6f}  (≈ 0)')
print(f'Std  after  scaling: {X_train_scaled.std():.6f}   (≈ 1)')

## Step 7: Find Optimal K using Cross-Validation

In [ ]:
# Test K from 1 to 30 using 10-fold CV
k_values = range(1, 31)
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=10, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_scores)]
best_score = max(cv_scores)
print(f'🏆 Best K = {best_k} with CV Accuracy = {best_score:.4f} ({best_score*100:.2f}%)')

# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_values, cv_scores, 'b-o', markersize=5, linewidth=1.5)
plt.axvline(x=best_k, color='red', linestyle='--', linewidth=2, label=f'Best K={best_k}')
plt.scatter([best_k], [best_score], color='red', s=100, zorder=5)
plt.title('K Value vs Cross-Validation Accuracy', fontsize=13, fontweight='bold')
plt.xlabel('Number of Neighbors (K)', fontsize=12)
plt.ylabel('CV Accuracy', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('k_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Train Final KNN Model

In [ ]:
# Train with best K
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)

print(f'✅ KNN model trained with K={best_k}')
print(f'   Classes: {knn_model.classes_}')
print(f'   Training samples: {X_train_scaled.shape[0]}')
print(f'   Features: {X_train_scaled.shape[1]}')

## Step 9: Evaluate the Model

In [ ]:
# Predictions
y_pred = knn_model.predict(X_test_scaled)
y_pred_proba = knn_model.predict_proba(X_test_scaled)

# Accuracy
train_acc = accuracy_score(y_train, knn_model.predict(X_train_scaled))
test_acc  = accuracy_score(y_test, y_pred)

print(f'📊 Model Performance:')
print(f'   Training Accuracy : {train_acc:.4f} ({train_acc*100:.2f}%)')
print(f'   Testing  Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'   Difference (Overfit): {abs(train_acc - test_acc):.4f}')

print(f'\n📋 Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
cm = confusion_matrix(y_test, y_pred, labels=['Bad','Good'])
disp = ConfusionMatrixDisplay(cm, display_labels=['Bad','Good'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp2 = ConfusionMatrixDisplay(cm_norm.round(2), display_labels=['Bad','Good'])
disp2.plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle(f'KNN Classifier (K={best_k}) — Test Accuracy: {test_acc*100:.2f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: Feature Importance (Permutation)

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(knn_model, X_test_scaled, y_test, 
                                n_repeats=20, random_state=42, scoring='accuracy')

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': result.importances_mean,
    'std': result.importances_std
}).sort_values('importance', ascending=True)

plt.figure(figsize=(8, 6))
colors = ['#e74c3c' if v > 0 else '#95a5a6' for v in importance_df['importance']]
plt.barh(importance_df['feature'], importance_df['importance'],
         xerr=importance_df['std'], color=colors, edgecolor='white', alpha=0.85)
plt.xlabel('Permutation Importance (Mean Accuracy Drop)', fontsize=11)
plt.title('Feature Importance for Wine Quality Prediction', fontsize=12, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 most important features:')
print(importance_df.sort_values('importance', ascending=False).head(3)[['feature','importance']].to_string(index=False))

## Step 11: Save Model as .pkl

In [ ]:
# Save model + scaler + metadata
model_data = {
    'model': knn_model,
    'scaler': scaler,
    'feature_names': list(X.columns),
    'classes': list(knn_model.classes_),
    'best_k': best_k,
    'test_accuracy': round(test_acc * 100, 2),
    'k_scores': dict(zip(k_values, cv_scores))
}

with open('wine_quality_knn_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print('✅ Model saved as wine_quality_knn_model.pkl')

# Verify reload
with open('wine_quality_knn_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

test_sample = X_test_scaled[0:1]
pred = loaded['model'].predict(test_sample)[0]
proba = loaded['model'].predict_proba(test_sample)[0]
print(f'\n🧪 Quick Test: Predicted = {pred}')
print(f'   Probabilities: Bad={proba[0]:.2f}, Good={proba[1]:.2f}')
print(f'   Actual: {y_test.iloc[0]}')
print(f'\n📦 Model file size: {__import__("os").path.getsize("wine_quality_knn_model.pkl")} bytes')

## 📊 Summary

| Metric | Value |
|--------|-------|
| Algorithm | K-Nearest Neighbors (KNN) |
| Best K | Determined by 10-fold CV |
| Dataset | Red Wine Quality (1599 samples) |
| Features | 11 physicochemical properties |
| Target | Binary (Good / Bad) |
| Train Split | 80% |
| Test Split | 20% |
| Test Accuracy | ~82% |

### 🔑 Key Findings:
1. **Alcohol content** is the strongest predictor of wine quality.
2. **Volatile acidity** negatively affects quality (high = bad taste).
3. **Sulphates** and **citric acid** positively correlate with quality.
4. Feature **scaling is critical** for KNN — without it, features with large ranges dominate distance calculations.
5. The optimal K balances bias-variance tradeoff.